# GPT Feature Database for Similarity

This notebook prompts GPT to extract concise feature vectors for each verdict and assembles a similarity dataset that mirrors the structure produced by `build_similarity_database.py`. Adjust the configuration section before running to point at your target and indictment facts CSV files.


In [6]:
import json
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator, Optional

import pandas as pd

from openai import OpenAI, OpenAIError


#free or law    
Extract = "law"
#drugs or wep
filed="drugs"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "REPLACED_OPENAI_KEY")


# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
if filed=="drugs":
    base_path="/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs"
else:
    base_path="/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon"

TARGET_PATH = Path(base_path,"target.csv")
FACTS_PATH = Path(base_path,"similarity_database_with_indicment_facts.csv")
if Extract == "free": 
    FEATURE_VECTORS_PATH = Path(base_path,"free_gpt_feature_vectors.csv")
    FINAL_DATABASE_PATH = Path(base_path,"similarity_database_with_gpt_features.csv")

else:
    FEATURE_VECTORS_PATH = Path(base_path,"law_gpt_feature_vectors.csv")
    FINAL_DATABASE_PATH = Path(base_path,"similarity_database_with_gpt_law_features.csv")

MODEL_NAME = os.getenv("GPT_FEATURE_MODEL", "gpt-4.1")
MAX_RETRIES = int(os.getenv("GPT_FEATURE_MAX_RETRIES", "3"))
RETRY_DELAY_SECONDS = float(os.getenv("GPT_FEATURE_RETRY_DELAY", "1.0"))
SAVE_EVERY = int(os.getenv("GPT_FEATURE_SAVE_EVERY", "10"))

# ----------------------------------------------------------------------
# FIX MODE Configuration
# ----------------------------------------------------------------------
# Options:
# - "normal": Start fresh or resume from existing (default behavior)
# - "empty_only": Re-extract only verdicts with empty feature vectors {}
# - "error_only": Re-extract verdicts with <3 features (potential errors)
# - "override_all": Re-extract ALL verdicts (overwrites everything)
# - "changed_only": Re-extract only verdicts that changed (from comparison CSV)
# ----------------------------------------------------------------------
FIX_MODE = "changed_only"

openai_client = OpenAI(api_key=OPENAI_API_KEY)
import os
# Delete the existing output file (only in normal mode)
if FIX_MODE == "normal" and os.path.exists(FEATURE_VECTORS_PATH):
    os.remove(FEATURE_VECTORS_PATH)
    print(f"Deleted {FEATURE_VECTORS_PATH}")

In [7]:
@dataclass(frozen=True)
class VerdictFacts:
    verdict: str
    indictment_facts: str


def ensure_directory(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def load_unique_verdict_facts(csv_path: Path) -> list[VerdictFacts]:
    """Collect a single record per verdict with the corresponding indictment facts."""
    df = pd.read_csv(csv_path)

    column_options: dict[str, list[str]] = {
        "verdict_1": ["verdict_1"],
        "verdict_2": ["verdict_2"],
        "facts_1": [
            "verdict_1_indictment_facts",
            "indicment_facts_1",
            "indictment_facts_1",
        ],
        "facts_2": [
            "verdict_2_indictment_facts",
            "indicment_facts_2",
            "indictment_facts_2",
        ],
    }

    resolved: dict[str, str] = {}
    for key, candidates in column_options.items():
        for candidate in candidates:
            if candidate in df.columns:
                resolved[key] = candidate
                break
        else:
            raise ValueError(f"Input CSV {csv_path} is missing columns {candidates}")

    frames: list[pd.DataFrame] = []
    for prefix in ("verdict_1", "verdict_2"):
        verdict_col = resolved[prefix]
        facts_key = "facts_1" if prefix == "verdict_1" else "facts_2"
        facts_col = resolved[facts_key]
        frame = (
            df[[verdict_col, facts_col]]
            .rename(columns={verdict_col: "verdict", facts_col: "indictment_facts"})
            .copy()
        )
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True)
    combined["verdict"] = combined["verdict"].astype(str).str.strip()
    combined["indictment_facts"] = combined["indictment_facts"].fillna("").astype(str).str.strip()
    combined = combined[combined["verdict"] != ""]

    combined = combined.drop_duplicates(subset="verdict", keep="first")
    verdicts = [
        VerdictFacts(verdict=row["verdict"], indictment_facts=row["indictment_facts"])
        for _, row in combined.iterrows()
    ]
    verdicts.sort(key=lambda item: item.verdict)
    return verdicts



In [8]:
if filed == "wep":
    law_part="""
   **בחוק העונשים פקודה מספר 144**:
  עבירות בנשק [א/66א

  144. (א)  המחזיק נשק בלא רשות על פי דין להחזקתו, דינו – מאסר שבע שנים, ואולם לעניין חלק מהותי בנשק – דינו מאסר חמש שנים, ולעניין חלק, אבזר או תחמושת כאמור בפסקאות (1) או (2) להגדרה "נשק", שאינם חלק מהותי בנשק (בסעיף זה – חלק לא מהותי בנשק), דינו – מאסר שלוש שנים.

  

            (ב)  הרוכש, נושא או מוביל נשק בלא רשות על פי דין לרכישתו, לנשיאתו או להובלתו, דינו – מאסר עשר שנים, ואולם לעניין חלק מהותי בנשק, דינו – מאסר חמש שנים, ולעניין חלק לא מהותי בנשק – מאסר שלוש שנים.

            (ב1)        סעיפים קטנים (א) ו-(ב) לא יחולו על מי שעבר את העבירות בשל כך בלבד שלא חידש את רישיונו או את תעודת הרשאתו, הכל בהתאם לחוק כלי היריה, תש"ט-1949, או התקנות על פיו.

            (ב2)        המייצר, מייבא או מייצא נשק או הסוחר בו או עושה בו כל עסקה אחרת שיש עמה מסירת החזקה בנשק לזולתו בין בתמורה ובין שלא בתמורה, בלא רשות על פי דין לעשות פעולה כאמור, דינו – מאסר חמש עשרה שנים.

            (ב3)        הרשאי על פי דין למכור או למסור נשק והוא מוכרו או מוסרו לאדם שאינו רשאי על פי דין להחזיק בו, דינו – מאסר חמש עשרה שנים; סבר המוכר או המוסר, כתוצאה מבדיקה רשלנית, שהוא מוכר או מוסר נשק למי שרשאי על פי דין להחזיק בו – דינו מאסר שלוש שנים.

            (ג)   בסעיף זה, "נשק" –

  (1)   כלי שסוגל לירות כדור, קלע, פגז, פצצה או כיוצא באלה, שבכוחם להמית אדם, וכולל חלק, אבזר ותחמושת של כלי כזה;

  (2)   כלי שסוגל לפלוט חומר הנועד להזיק לאדם, לרבות חלק, אבזר ותחמושת לכלי כאמור ולרבות מכל המכיל או שסוגל להכיל חומר כאמור ולמעט מכל גז מדמיע כהגדרתו בחוק כלי היריה, תש"ט-1949;

  (3)   תחמושת, פצצה, רימון או כל חפץ נפיץ אחר שבכוחם להמית אדם או להזיק לו, לרבות חלק של אחד מאלה.

            "חלק מהותי בנשק" – חלק או אבזר כאמור בפסקאות (1) או (2) להגדרה "נשק" שהוא גוף, קנה, צינה, מכלול או סדן של כלי נשק.

            (ג1) לענין סעיף זה –

  (1)   אחת היא אם בעת שנעברה העבירה היה הנשק תקין לשימוש או לא;

  (2)   הטוען לרשות על פי דין - עליו הראיה.

            (ד)  מקום שנמצא בו נשק, רואים את מחזיק המקום כמחזיק הנשק כל עוד לא הוכח היפוכו של דבר.

            (ה)  תעודה החתומה בידי קצין משטרה בדרגת מפקח ומעלה והיא מאשרת שחפץ פלוני הוא נשק, תשמש ראיה לדבר, כל עוד לא הוכח היפוכו; אולם הנאשם זכאי להזמין את חותם התעודה לחקירה, ואם עשה כן, לא תשמש התעודה ראיה אלא אם חותם התעודה נענה להזמנה; בית המשפט חייב להודיע לנאשם על זכותו להזמין את חותם התעודה לחקירה.

            (ו)   סעיף זה אינו בא לגרוע מהוראת כל דין.

            (ז)   הורשע אדם בעבירה לפי סעיף קטן (א) רישה, (ב) רישה, (ב2) או (ב3) רישה, לא יפחת עונשו מרבע העונש המרבי שנקבע לאותה עבירה, אלא אם כן החליט בית המשפט, מטעמים מיוחדים שיירשמו, להקל בעונשו; עונש מאסר לפי סעיף קטן זה לא יהיה, בהעדר טעמים מיוחדים, כולו על-תנאי.

            (ח)  הורשע אדם בעבירה לפי סעיף קטן (ב2), יצווה בית המשפט, זולת אם סבר שלא לעשות כן מנימוקים מיוחדים שיפרט, כי נוסף על כל עונש יחולט לאוצר המדינה כל רכוש שהוא אחד מאלה:

  (1)   רכוש ששימש או נועד לשמש אמצעי לביצוע העבירה או ששימש או נועד לשמש כדי לאפשר את ביצוע העבירה;

  (2)   רכוש שהושג, במישרין או בעקיפין, כשכר העבירה או כתוצאה מביצוע העבירה או שיועד לכך.

            (ט)  הורשע אדם בעבירה לפי סעיף קטן (ב2) והתקיים אחד מאלה, רשאי בית המשפט לקבוע, לבקשת תובע, כי הנידון ניהל אורח חיים המבוסס על שימוש בתקבולי עבירה:

  (1)   העבירה בוצעה לגבי יותר מנשק אחד;

  (2)   בשש השנים שקדמו להרשעה באותה עבירה, הורשע הנידון בעבירה נוספת לפי סעיף קטן (ב2);

  (3)   העבירה בוצעה בזיקה לארגון פשיעה כהגדרתו בחוק מאבק בארגוני פשיעה, התשס"ג-2003;

  (4)   שווי תקבולי העבירה הוא 50,000 שקלים חדשים או יותר.

            (י)   קבע בית המשפט כאמור בסעיף קטן (ט), יצווה בגזר הדין כי נוסף על כל עונש יחולט לאוצר המדינה כל רכוש של הנידון שהושג בעבירה לפי סעיף קטן (ב2), אלא אם כן סבר שלא לעשות כן מנימוקים מיוחדים שיפרט, ויחולו לעניין זה הוראות סעיף 31(6) לפקודת הסמים המסוכנים [נוסח חדש], התשל"ג-1973, בשינויים המחויבים.

            (יא) על חילוט רכוש לפי סעיף זה יחולו הוראות לפי סעיפים 36א(ג) עד (ו), 36ב עד 36ז ו-36ט לפקודת הסמים המסוכנים [נוסח חדש], התשל"ג-1973, בשינויים המחויבים ובשינויים אלו:

  (1)   בסעיף 36ב(א) –

  (א)   בפסקה (1), במקום "לפי סעיפים 6 או 13" יקראו "לפי סעיף 144(ב2) לחוק העונשין, התשל"ז-1977";

  (ב)   בפסקאות (2) ו-(3), בכל מקום, במקום "עבירה של עסקת סמים" יקראו "עבירה לפי סעיף 144(ב2) לחוק העונשין, התשל"ז-1977";

  (2)   בסעיף 26ז, במקום "כאמור בסעיפים 36א או 36ב" יקראו "כאמור בסעיף 144(ח)(2) לחוק העונשין, התשל"ז-1977".

"""

if filed == "drugs":
    law_part= """
להלן עיקרי הוראות החוק הרלוונטיות (פקודת הסמים המסוכנים ):

פרק ג':עבירות
סימן א':ייצור, החזקה ושימוש
6.	לא יגדל אדם סם מסוכן, לא ייצר אותו, לא יפיק אותו, לא יכין אותו ולא ימצה אותו מחומר אחר, אלא ברשיון מאת המנהל; העובר על הוראות סעיף זה, דינו – מאסר עשרים שנים או קנס פי עשרים וחמישה מן הקנס האמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977.
7.	(א)	לא יחזיק אדם סם מסוכן ולא ישתמש בו, אלא במידה שהותר הדבר בפקודה זו או בתקנות לפיה, או ברשיון מאת המנהל.
(ב)	האמור בסעיף זה בדבר איסור החזקה אינו חל על סם מסוכן הנמצא במעבר שהותר לפי פקודה זו.
(ג)	העובר על הוראות סעיף זה, דינו – מאסר עשרים שנים או קנס פי עשרים וחמישה מן הקנס האמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977, ואם החזיק בסם או השתמש בו לצריכתו העצמית בלבד, דינו – מאסר שלוש שנים או קנס כאמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977.
(ג1)	(פקע)
(ד)	על אף הוראות סעיף קטן (ג) סיפה, מי, שלצריכתו העצמית, מחזיק בסם או משתמש בו בתוך כותלי בית ספר או בחצריו והוא אינו לומד באותו בית ספר, דינו – מאסר חמש שנים; הוראה זו לא תחול על מי שטרם מלאו לו שש עשרה שנים.
8.	לענין אישום בשל החזקת סם מסוכן, אין נפקא מינה אם הסם המסוכן נמצא ברשותו של הנאשם, או ברשות המחזיק אותו מטעמו של הנאשם, או אם הסם של הנאשם נמצא ברשותו של אדם אחר ללא ידיעתו של אותו אחר, או אם הסם נמצא במקום שאינו ברשותו או שאינו נתון לפיקוחו או להשגחתו של שום אדם.
9.	(א)	המחזיק חצרים לא ירשה להשתמש בהם לשם הכנת סם מסוכן, שימוש בו, מכירתו או עשיית עסקה אחרת בו, שלא בהיתר.
(ב)	לא יהיה לאדם ענין בניהול חצרים המשמשים למטרה כאמור בסעיף קטן (א).
(ג)	לא יהיה אדם נוהג לבקר במקום שנועד לשימוש בסמים מסוכנים.
(ד)	העובר על הוראות סעיף זה, דינו – מאסר עשרים שנים או קנס פי עשרים וחמישה מן הקנס האמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977, ואם עבר עליהן אגב החזקת סם מסוכן או שימוש בו לצריכתו העצמית בלבד, דינו – מאסר שלוש שנים או קנס כאמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977.
10.	לא יהיו ברשותו של אדם כלים המשמשים להכנת סם מסוכן או לצריכתו, שלא בהיתר; העובר על הוראות סעיף זה, דינו – מאסר עשרים שנים או קנס פי עשרים וחמישה מהקנס האמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977, ואם היו הכלים לשימוש בצריכתו העצמית בלבד, דינו – מאסר שלוש שנים או קנס כאמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977.
10א.	(א)	לא ייצר אדם כלי אסור, לא ימכרו, לא יציגו למכירה, לא ייבאו, לא ייצאו, לא יסחר בו או יעשה בו עסקה אחרת ולא יחזיק בו במטרה לעשות אחת מהפעולות האמורות, שלא בהיתר; העובר על הוראות סעיף זה, דינו – מאסר חמש שנים.
(ב)	בסעיף זה, "כלי אסור" – כלי שייעודו העיקרי, דרך כלל, הוא להכנת סם מסוכן או לצריכתו, כגון הכלים המנויים להלן:
(1)	באנג.
11.	לענין פקודה זו, החזקת סם כמפורט בחלק ב' לתוספת הראשונה מותרת באחת מאלה:
(1)	המחזיק הוא רוקח והסם מוחזק בחצריו שיש לו רשיון עליהם;
(2)	המחזיק הוא רופא, רופא שיניים, רופא וטרינר, עמית רופא או אח או אחות מומחים, ולפי חיקוק הדן ברופאים, ברופאי שיניים, ברופאים וטרינרים, בעמיתי רופא או באח או אחות מומחים מותר לו להחזיק סם כאמור;
(3)	המחזיק מוכיח שהוא השיג את הסם שבהחזקתו מרוקח ושהסם נופק לפי הוראות פקודת הרוקחים, או שהושג מרופא, מרופא וטרינר, מעמית רופא או מאח או אחות מומחים הרשאים לפי הדין לספק סמים או תרופות;
(4)	הדבר הורשה בתקנות לפי פקודה זו.
12.	השימוש בסם מסוכן מותר אם הוא לצורך ריפוי והסם סופק למשתמש מאת רוקח, רופא, רופא וטרינר, עמית רופא או אח או אחות מומחים בתנאים האמורים בסעיף 11(3) או סופק על פי רשיון.
סימן ב':מסחר ומעבר
13.	לא ייצא אדם סם מסוכן, לא ייבא אותו, לא יקל על ייצואו או ייבואו, לא יסחר בו, לא יעשה בו שום עסקה אחרת ולא יספקנו בשום דרך בין בתמורה ובין שלא בתמורה, אלא אם הותר הדבר בפקודה זו או בתקנות לפיה או ברשיון מאת המנהל.
14.	לא יתווך אדם – בין בתמורה ובין שלא בתמורה – בפעולה אסורה לפי סעיף 13.
15.	לא יוביל אדם סם מסוכן במעבר דרך ישראל אלא מארץ שמותר לייצאו ממנה ואל ארץ אחרת שמותר לייבאו אליה; בא הסם מארץ שהיא מבעלות האמנה – תנאי נוסף הוא שיהא עם הסם היתר יצוא או היתר הטיה בר-תוקף.
16.	(א)	סם מסוכן שהובא לישראל במעבר, לא יגרום אדם להטייתו למקום יעוד שאינו המקום שאליו נשגר מתחילה, אלא על פי היתר הטיה.
(ב)	סם במעבר, שיש עמו היתר יצוא או היתר הטיה מאת רשות מוסמכת של ארץ חוץ, יראו את ארץ היעוד לפי האמור בהיתר כארץ אשר אליה נשגר הסם מתחילה.
17.	(א)	לא ירחיק אדם סם מסוכן מהרכב שבו הובא לישראל במעבר, ולא יטלטל אדם סם מסוכן בישראל לאחר שהורחק כאמור, אלא לפי רשיון הרחקה מאת מנהל אגף המכס והבלו.
(ב)	נתינתו וסירובו של רשיון הרחקה כאמור מסורים לשיקול דעתו המוחלט של מנהל אגף המכס והבלו.
18.	סם מסוכן שבמעבר, לא יתננו אדם לתהליך העשוי לשנותו שינוי מהותי ולא יפתח ולא ישבור, במזיד, אריזה המכילה אותו, אלא לפי הוראות המנהל ובדרך שהורה.
19.	הוראות סעיפים 15 עד 18 לא יחולו על –
(1)	סם מסוכן במעבר בדואר;
(2)	סם מסוכן במעבר בכלי טיס העובר בשמי ישראל ואינו נוחת בה;
(3)	כמות של סם מסוכן היכולה להיות, בתום לב ובסבירות, חלק מן המלאי הרפואי של כלי שיט או כלי טיס.
19א.	העובר על הוראות סימן זה, דינו – מאסר עשרים שנים או קנס פי עשרים וחמישה מהקנס האמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977.
20.	בכפוף לסייג כאמור בסעיף 19, רשאי מנהל אגף המכס והבלו, או פקיד שהוא הסמיך, לדרוש שיוצג היתר היצוא או היתר ההטיה הנוגע למשגור של סם מסוכן המובל במעבר בישראל, ורשאי הוא לנקוט פעולה נוספת לגבי המשגור כפי שנקבע בתקנות.
סימן ג':הדחת קטינים
21.	(א)	העושה אחת מאלה, דינו – מאסר עשרים וחמש שנים או קנס פי עשרים וחמישה מהקנס האמור בסעיף 61(א)(4) לחוק העונשין, התשל"ז-1977:
(1)	נותן לקטין סם מסוכן;
(2)	בהיותו אחראי לקטין מניח לו להשיג סם מסוכן או להשתמש בו;
(3)	משדל קטין להשיג סם מסוכן או להשתמש בו.
(ב)	לענין סעיף זה, "אחראי לקטין" – הורה, לרבות הורה חורג, מאמץ, אפוטרופוס או מי שהקטין נמצא במשמורתו או בהשגחתו.
22.	סעיף 21 לא יחול על מי שעשה את המעשה לשם טיפול רפואי, בהיותו רופא, רופא שיניים, עמית רופא או אח או אחות מומחים, או לפי הוראת רופא, רופא שיניים, עמית רופא או אח או אחות מומחים, או בנסיבות חוקיות אחרות.

"""



In [9]:
import json
import re

def sanitise_json_text(text: str) -> dict:
    """
    Robustly cleans and parses JSON from LLM output.
    Handles Markdown blocks, control characters, and Hebrew text.
    """
    if not text:
        return {}

    # 1. Remove Markdown code blocks (```json ... ```)
    text = re.sub(r'^```json\s*', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'^```\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s*```$', '', text, flags=re.MULTILINE)

    # 2. Extract the JSON object (find first { and last })
    start_idx = text.find('{')
    end_idx = text.rfind('}')
    
    if start_idx == -1 or end_idx == -1:
        # Fallback: return empty or try manual extraction if needed
        return {}
    
    json_str = text[start_idx : end_idx + 1]

    # 3. Clean "Invalid Control Characters" (Newlines/Tabs inside strings)
    # This regex looks for newlines that are NOT structural JSON newlines
    # (A simple approach is to just allow them via strict=False in standard json.loads)
    
    try:
        # strict=False allows control characters like \n inside strings
        return json.loads(json_str, strict=False)
    except json.JSONDecodeError:
        try:
            # common error: trailing commas before closing brace (e.g., "key": "val", })
            json_str = re.sub(r',\s*}', '}', json_str)
            return json.loads(json_str, strict=False)
        except json.JSONDecodeError:
            # common error: unescaped quotes inside strings. 
            # It's hard to fix perfectly with regex, but we can try removing distinct bad escapes
            pass
            
    return {} # Failed to parse

def parse_feature_json(text: str) -> dict[str, object]:
    """Parse JSON from GPT response with multiple fallback strategies."""
    import re
    
    cleaned = sanitise_json_text(text)
    
    # If sanitise_json_text already returned a dict, use it directly
    if isinstance(cleaned, dict):
        return cleaned
    
    # Try standard parsing first
    try:
        parsed = json.loads(cleaned)
        if not isinstance(parsed, dict):
            raise ValueError("Expected a JSON object with key/value pairs")
        return parsed
    except json.JSONDecodeError as e:
        # Log the problematic JSON for debugging
        print(f"\n⚠️  JSON parsing error: {e}")
        print(f"Problematic JSON snippet (chars {max(0, e.pos-50)}:{min(len(cleaned), e.pos+50)}):")
        print(f"...{cleaned[max(0, e.pos-50):min(len(cleaned), e.pos+50)]}...")
        
        # Try to fix common issues and retry
        # 1. Remove comments (if any)
        cleaned_no_comments = re.sub(r'//.*?$', '', cleaned, flags=re.MULTILINE)
        cleaned_no_comments = re.sub(r'/\*.*?\*/', '', cleaned_no_comments, flags=re.DOTALL)
        
        try:
            parsed = json.loads(cleaned_no_comments)
            if not isinstance(parsed, dict):
                raise ValueError("Expected a JSON object with key/value pairs")
            print("✓ Fixed by removing comments")
            return parsed
        except json.JSONDecodeError:
            pass
        
        # 2. Try to fix missing commas between properties
        # This is a heuristic approach - look for patterns like: "value"\n"key":
        fixed = re.sub(r'"\s*\n\s*"', '",\n"', cleaned)
        fixed = re.sub(r'(\d+)\s*\n\s*"', r'\1,\n"', fixed)
        fixed = re.sub(r'(true|false|null)\s*\n\s*"', r'\1,\n"', fixed, flags=re.IGNORECASE)
        
        try:
            parsed = json.loads(fixed)
            if not isinstance(parsed, dict):
                raise ValueError("Expected a JSON object with key/value pairs")
            print("✓ Fixed by adding missing commas")
            return parsed
        except json.JSONDecodeError:
            pass
        
        # 3. Last resort: try to extract key-value pairs manually
        print("⚠️  Attempting manual key-value extraction as last resort...")
        try:
            # Use more lenient regex patterns to extract key-value pairs
            # Pattern 1: Quoted keys with quoted string values
            pattern1 = r'"([^"]+)"\s*:\s*"([^"]*)"'
            # Pattern 2: Unquoted keys with quoted string values
            pattern2 = r'([א-תa-zA-Z_][א-תa-zA-Z0-9_]*)\s*:\s*"([^"]*)"'
            # Pattern 3: Quoted keys with numeric values
            pattern3 = r'"([^"]+)"\s*:\s*([-+]?\d+\.?\d*)'
            # Pattern 4: Unquoted keys with numeric values
            pattern4 = r'([א-תa-zA-Z_][א-תa-zA-Z0-9_]*)\s*:\s*([-+]?\d+\.?\d*)'
            
            result = {}
            for pattern in [pattern1, pattern2, pattern3, pattern4]:
                matches = re.findall(pattern, cleaned)
                for key, value in matches:
                    if key not in result:  # Don't overwrite existing keys
                        # Try to convert numeric strings to numbers
                        try:
                            if '.' in str(value):
                                result[key] = float(value)
                            else:
                                result[key] = int(value)
                        except (ValueError, TypeError):
                            result[key] = value
            
            if result:
                print(f"✓ Extracted {len(result)} key-value pairs manually")
                return result
        except Exception as manual_error:
            print(f"Manual extraction also failed: {manual_error}")
        
        # Re-raise the original error with the cleaned text
        print(f"\nFull cleaned JSON that failed to parse:")
        print(cleaned)
        raise ValueError(f"Could not parse JSON after multiple attempts: {e}") from e


def build_user_prompt(verdict_facts: VerdictFacts) -> str:
    return USER_PROMPT_TEMPLATE.format(
        verdict_id=verdict_facts.verdict,
        facts=verdict_facts.indictment_facts,
        law_part=law_part,
    )


def request_features(
    client: OpenAI,
    verdict_facts: VerdictFacts,
    *,
    model: str,
    max_retries: int,
    retry_delay: float,
) -> dict[str, object]:
    last_error: Optional[Exception] = None
    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=model,
                temperature=0,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(verdict_facts)},
                ],
            )
            content = completion.choices[0].message.content or ""
            return parse_feature_json(content)
        except (OpenAIError, json.JSONDecodeError, ValueError) as exc:
            last_error = exc
            if attempt == max_retries:
                break
            time.sleep(retry_delay * attempt)
    assert last_error is not None
    raise RuntimeError(
        f"Failed to extract features for verdict {verdict_facts.verdict!r} after {max_retries} attempts"
    ) from last_error



In [10]:
SYSTEM_PROMPT = (
"""
אתה אנליסט משפטי מומחה המתמחה במשפט הפלילי הישראלי.
אתה מנתח סיכומי פסקי דין כדי לחלץ מאפיינים מובנים התומכים בניקוד דמיון בין תיקים.

חשוב מאוד לגבי פורמט הפלט:
- החזר JSON תקני בלבד, אובייקט אחד {}.
- כל מפתחות (keys) חייבים להיות במרכאות כפולות.
- כל ערכי מחרוזת חייבים להיות במרכאות כפולות.
- אין לשבור שורות בתוך ערכי מחרוזת.
- השתמש רק בתווים תקניים - אין תווי בקרה כמו newline או tab בתוך מחרוזות.
""")

if Extract == "free": 

    USER_PROMPT_TEMPLATE = """
    אתה מומחה לדין הפלילי הישראלי.

     תפקידך:
  אתה מנתח את עובדות כתב האישום מפסק הדין ומחלץ רשימת תכונות (פיצ׳רים) מובניים התומכים בניקוד דמיון בין תיקים
  כל מפתח מייצג תכונה רלוונטית לעבירה או לעונש, והערך מתאר את הנתון כפי שעולה מפסק הדין.
    עבור כל שדה, ציין את התווית והערך שלו
    
    
דרישות קריטיות לפלט (Critical JSON Rules):
1. החזר אך ורק **JSON תקין** (Valid JSON Object).
2. **אל תשתמש** בסימון Markdown (בלי ```json).
3. אל תכתוב הקדמות או הסברים. רק את ה-JSON.
4. וודא שכל המפתחות והערכים מוקפים במרכאות כפולות (").
5. אל תשבור שורות באמצע ערך של מחרוזת (No newlines inside strings).
חשוב מאוד:
במידה ויש פרטים בנוגע לעונש, להסדר טיעון או להכרעת הדין - אל תכלול אותם בפיצ'רים!

עובדות עיקריות:
{facts}

"""
else:
     
    #   לחלץ רשימת תכונות (פיצ'רים) מהחלק של עובדות כתב האישום בפסקי דין, בפורמט של מילון בפורמט json.
    USER_PROMPT_TEMPLATE = """
אתה מומחה לדין הפלילי בישראל.

  תפקידך:
  אתה מנתח את עובדות כתב האישום מפסק הדין ומחלץ רשימת תכונות (פיצ׳רים) מובניים התומכים בניקוד דמיון בין תיקים
  כל מפתח מייצג תכונה רלוונטית לעבירה או לעונש, והערך מתאר את הנתון כפי שעולה מפסק הדין.
    עבור כל שדה, ציין את התווית והערך שלו

  עליך להסתמך על שני מקורות עיקריים:

  1. **הנסיבות הקשורות בביצוע העבירה** (חוק העונשין, סעיף 40ט):
  40ט.  (א)  בקביעת מתחם העונש ההולם למעשה העבירה שביצע הנאשם כאמור בסעיף 40ג(א), יתחשב בית המשפט בהתקיימותן של נסיבות הקשורות בביצוע העבירה, המפורטות להלן, ובמידה שבה התקיימו, ככל שסבר שהן משפיעות על חומרת מעשה העבירה ועל אשמו של הנאשם:

  (1)   התכנון שקדם לביצוע העבירה;

  (2)   חלקו היחסי של הנאשם בביצוע העבירה ומידת ההשפעה של אחר על הנאשם בביצוע העבירה;

  (3)   הנזק שהיה צפוי להיגרם מביצוע העבירה;

  (4)   הנזק שנגרם מביצוע העבירה;

  (5)   הסיבות שהביאו את הנאשם לבצע את העבירה;

  (6)   יכולתו של הנאשם להבין את אשר הוא עושה, את הפסול שבמעשהו או את משמעות מעשהו, לרבות בשל גילו;

  (7)   יכולתו של הנאשם להימנע מהמעשה ומידת השליטה שלו על מעשהו, לרבות עקב התגרות של נפגע העבירה;

  (8)   מצוקתו הנפשית של הנאשם עקב התעללות בו על ידי נפגע העבירה;

  (9)   הקרבה לסייג לאחריות פלילית כאמור בסימן ב' לפרק ה'1;

  (10)  האכזריות, האלימות וההתעללות של הנאשם בנפגע העבירה או ניצולו;

  (11)  הניצול לרעה של כוחו או מעמדו של הנאשם או של יחסיו עם נפגע העבירה.

  עלייך להתחשב בנוסף בנסיבות שאינן קשורות לביצוע העבירה לפי החוק :



  3.{law_part}
  
דרישות קריטיות לפלט:
- החזר/י **JSON תקני בלבד** (אובייקט אחד) ללא שום טקסט נוסף.
- אין הסברים, אין פרוזה, אין שדות מעבר לסכמה.
- חשוב מאוד: אל תשתמש בסימני מרכאות (") בתוך שמות המפתחות (keys).
  במקום "כמות_ק"ג" השתמש ב-"כמות_קג" או "כמות_בקילוגרם".
חשוב מאוד:
במידה ויש פרטים בנוגע לעונש, להסדר טיעון או להכרעת הדין - אל תכלול אותם בפיצ'רים!

עובדות עיקריות:
{facts}
  

"""
    


In [11]:
def iter_pending_verdicts(
    all_verdicts: Iterable[VerdictFacts],
    processed: set[str],
) -> Iterator[VerdictFacts]:
    for item in all_verdicts:
        if item.verdict not in processed:
            yield item


def load_existing_output(output_path: Path) -> Optional[pd.DataFrame]:
    if not output_path.exists():
        return None
    df = pd.read_csv(output_path)
    df["verdict"] = df["verdict"].astype(str).str.strip()
    return df


def save_feature_vectors(records: list[dict[str, object]], output_path: Path) -> pd.DataFrame:
    df = pd.DataFrame(records)
    df = df[["verdict", "indictment_facts", "feature_vector_json"]]
    df = df.sort_values("verdict").reset_index(drop=True)
    ensure_directory(output_path)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    return df


def extract_feature_vectors(
    client: OpenAI,
    verdicts: list[VerdictFacts],
    *,
    output_path: Path,
    model: str,
    max_retries: int,
    retry_delay: float,
    save_every: int,
) -> pd.DataFrame:
    existing_df = load_existing_output(output_path)
    results: list[dict[str, object]] = []
    processed: set[str] = set()

    if existing_df is not None:
        processed = set(existing_df["verdict"])
        results.extend(existing_df.to_dict(orient="records"))
        print(f"Resuming from {len(processed)} already processed verdicts.")

    pending = list(iter_pending_verdicts(verdicts, processed))
    print(f"Processing {len(pending)} remaining verdicts with model '{model}'...")

    for idx, verdict_facts in enumerate(pending, start=1):
        if verdict_facts.indictment_facts:
            feature_payload = request_features(
                client,
                verdict_facts,
                model=model,
                max_retries=max_retries,
                retry_delay=retry_delay,
            )
        else:
            feature_payload = {}

        results.append(
            {
                "verdict": verdict_facts.verdict,
                "indictment_facts": verdict_facts.indictment_facts,
                "feature_vector_json": json.dumps(feature_payload, ensure_ascii=False, sort_keys=True),
            }
        )

        if idx % save_every == 0:
            save_feature_vectors(results, output_path)
            print(f"  Saved checkpoint after {idx}/{len(pending)} verdicts.")

        if retry_delay > 0:
            time.sleep(retry_delay)

    final_df = save_feature_vectors(results, output_path)
    print(f"Completed feature extraction for {len(results)} verdicts.")
    print(f"Feature vectors saved to {output_path}")
    return final_df



In [12]:
verdict_facts_list = load_unique_verdict_facts(FACTS_PATH)
print(f"Identified {len(verdict_facts_list)} unique verdicts in the indictment facts dataset.")

# Apply FIX_MODE logic
if FIX_MODE != "normal":
    print(f"\n🔧 FIX MODE: {FIX_MODE}")
    print("="*70)
    
    existing_df = load_existing_output(FEATURE_VECTORS_PATH)
    if existing_df is None:
        print("⚠️  No existing file found. Running in normal mode.")
        FIX_MODE = "normal"
    else:
        print(f"Loaded {len(existing_df)} existing feature vectors")
        
        # Determine which verdicts to re-extract
        if FIX_MODE == "empty_only":
            mask = existing_df['feature_vector_json'] == '{}'
            empty_verdicts = set(existing_df[mask]['verdict'])
            print(f"Found {len(empty_verdicts)} verdicts with empty features")
            
            # Filter verdict_facts_list to only include empty verdicts
            verdict_facts_list = [v for v in verdict_facts_list if v.verdict in empty_verdicts]
            
        elif FIX_MODE == "error_only":
            def count_features(json_str):
                try:
                    return len(json.loads(json_str))
                except:
                    return 0
            existing_df['feature_count'] = existing_df['feature_vector_json'].apply(count_features)
            mask = (existing_df['feature_count'] < 3) & (existing_df['indictment_facts'].notna())
            error_verdicts = set(existing_df[mask]['verdict'])
            print(f"Found {len(error_verdicts)} verdicts with <3 features")
            
            # Filter verdict_facts_list to only include error verdicts
            verdict_facts_list = [v for v in verdict_facts_list if v.verdict in error_verdicts]
            
        elif FIX_MODE == "changed_only":
            # Load comparison CSV to get changed verdict IDs
            comparison_csv_path = Path(base_path, "similarity_database_with_indicment_facts_comparison_dry_run.csv")
            
            if not comparison_csv_path.exists():
                print(f"⚠️  Comparison CSV not found at {comparison_csv_path}")
                print("   Please run update_similarity_database_with_indicment_facts_with_new_extracted.py first")
                FIX_MODE = "normal"
            else:
                print(f"📂 Loading comparison CSV: {comparison_csv_path}")
                df_changes = pd.read_csv(comparison_csv_path)
                
                # Get unique changed verdict IDs
                changed_verdicts = set(df_changes["verdict_id"].astype(str).str.strip())
                print(f"Found {len(changed_verdicts)} unique verdicts with changes")
                
                # Filter verdict_facts_list to only include changed verdicts
                verdict_facts_list = [v for v in verdict_facts_list if v.verdict in changed_verdicts]
                
                # Remove these verdicts from processed set so they get re-processed
                if existing_df is not None:
                    # Remove changed verdicts from the processed set
                    existing_df = existing_df[~existing_df["verdict"].isin(changed_verdicts)]
                    # Save updated existing_df (without changed verdicts)
                    from pathlib import Path
                    ensure_directory(FEATURE_VECTORS_PATH)
                    existing_df.to_csv(FEATURE_VECTORS_PATH, index=False, encoding="utf-8-sig")
                    print(f"   Saved updated feature vectors (removed changed verdicts)")
                    print(f"   Removed {len(changed_verdicts)} changed verdicts from existing data (will be re-processed)")
            
        elif FIX_MODE == "override_all":
            print(f"⚠️  Will re-extract ALL {len(verdict_facts_list)} verdicts!")
        
        print(f"Will process {len(verdict_facts_list)} verdicts")
        print("="*70 + "\n")

feature_vectors_df = extract_feature_vectors(
    openai_client,
    verdict_facts_list,
    output_path=FEATURE_VECTORS_PATH,
    model=MODEL_NAME,
    max_retries=MAX_RETRIES,
    retry_delay=RETRY_DELAY_SECONDS,
    save_every=SAVE_EVERY,
)

feature_vectors_df.head()


Identified 68 unique verdicts in the indictment facts dataset.

🔧 FIX MODE: changed_only
Loaded 47 existing feature vectors
📂 Loading comparison CSV: /Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/similarity_database_with_indicment_facts_comparison_dry_run.csv
Found 21 unique verdicts with changes
   Saved updated feature vectors (removed changed verdicts)
   Removed 21 changed verdicts from existing data (will be re-processed)
Will process 21 verdicts

Resuming from 47 already processed verdicts.
Processing 21 remaining verdicts with model 'gpt-4.1'...


KeyboardInterrupt: 

In [ ]:
def prepare_similarity_frame(target_path: Path) -> pd.DataFrame:
    df = pd.read_csv(target_path)

    required_columns = {"verdict_1", "verdict_2", "similarity"}
    missing = required_columns.difference(df.columns)
    if missing:
        raise ValueError(
            f"Target CSV {target_path} is missing required columns: {sorted(missing)}"
        )

    df["verdict_1"] = df["verdict_1"].astype(str).str.strip()
    df["verdict_2"] = df["verdict_2"].astype(str).str.strip()

    df["similarity_scale"] = df["similarity"]

    binary_0_map = {1: 0, 2: 0, 3: 1}
    binary_1_map = {1: 0, 2: 1, 3: 1}

    invalid_values = sorted(
        {value for value in df["similarity_scale"].unique() if value not in binary_0_map}
    )
    if invalid_values:
        raise ValueError(
            f"Unexpected similarity values {invalid_values} found in {target_path}."
        )

    df["similarity_binary_0"] = df["similarity_scale"].map(binary_0_map).astype(int)
    df["similarity_binary_1"] = df["similarity_scale"].map(binary_1_map).astype(int)
    return df


def build_similarity_dataset_with_gpt_features(
    base_df: pd.DataFrame,
    feature_vectors: pd.DataFrame,
    output_path: Path,
) -> pd.DataFrame:
    feature_map = feature_vectors.set_index("verdict")["feature_vector_json"].to_dict()

    output_columns = [
        "verdict_1",
        "verdict_2",
        "similarity_scale",
        "similarity_binary_0",
        "similarity_binary_1",
    ]

    output_df = base_df[output_columns].copy()
    output_df["feature_vector_1"] = output_df["verdict_1"].map(feature_map).fillna("{}")
    output_df["feature_vector_2"] = output_df["verdict_2"].map(feature_map).fillna("{}")

    ensure_directory(output_path)
    output_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    missing_v1 = (output_df["feature_vector_1"] == "{}").sum()
    missing_v2 = (output_df["feature_vector_2"] == "{}").sum()

    print(f"Saved GPT feature similarity database to {output_path}")
    print(f"Missing feature vectors -> verdict_1: {missing_v1}, verdict_2: {missing_v2}")

    return output_df



In [ ]:
similarity_base_df = prepare_similarity_frame(TARGET_PATH)
print(f"Loaded {len(similarity_base_df)} verdict pairs from the target dataset.")

gpt_similarity_df = build_similarity_dataset_with_gpt_features(
    similarity_base_df,
    feature_vectors_df,
    FINAL_DATABASE_PATH,
)

gpt_similarity_df.head()


Loaded 100 verdict pairs from the target dataset.
Saved GPT feature similarity database to /Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/similarity_database_with_gpt_law_features.csv
Missing feature vectors -> verdict_1: 0, verdict_2: 0


,verdict_1,verdict_2,similarity_scale,similarity_binary_0,similarity_binary_1,feature_vector_1,feature_vector_2
0,ME-16-07-11608-225,ME-22-12-47987-640,1,0,0,"{""אמצעי_הצפנה"": ""אפליקציית ויקר"", ""האם_הנאשם_ה...","{""ארץ_מוצא"": ""הודו"", ""האם_בוצעה_מסירה_מבוקרת"":..."
1,ME-16-11-63255-338,ME-18-01-73652-357,1,0,0,"{""אכזריות_או_אלימות"": ""לא צוין"", ""היכרות_מוקדמ...","{""אופי_התקשורת"": ""הודעות ווטסאפ, שיחות טלפון, ..."
2,ME-16-12-26620-433,ME-16-06-6788-21,1,0,0,"{""אופן_העברת_הסם"": ""הוסתר במזוודה שנמסרה לנאשמ...","{""אופן_ההסלקה"": ""הוסלק בשרוולי מעילים"", ""אופן_..."
3,ME-17-01-59971-328,ME-14-01-16958-980,3,1,1,"{""אכזריות_או_ניצול"": false, ""היקף_הפעילות"": ""מ...","{""אכזריות_או_ניצול"": ""לא צוינה"", ""האם_החזיק_כל..."
4,ME-17-01-620-279,ME-16-12-19902-389,3,1,1,"{""אופן_היבוא"": ""דרך טיסה בינלאומית"", ""אכזריות_...","{""אופן_קבלת_הסם"": ""המזוודה נמסרה לנאשם על ידי ..."


In [ ]:
# ========================================================================
# FIX EMPTY/ERROR FEATURE VECTORS
# ========================================================================

# Configuration flag:
# - "empty_only": Re-extract only verdicts with empty feature vectors {}
# - "error_only": Re-extract only verdicts with parsing errors (future use)
# - "override_all": Re-extract all verdicts (WARNING: overwrites everything!)
# - "skip": Don't run this cell (default)

FIX_MODE = "empty_only"  # Change to "empty_only", "error_only", or "override_all" to run

# ========================================================================

if FIX_MODE != "skip":
    print(f"🔧 Running FIX MODE: {FIX_MODE}")
    print("="*70)
    
    # Load current feature vectors
    existing_df = load_existing_output(FEATURE_VECTORS_PATH)
    
    if existing_df is None:
        print("❌ No existing feature vectors file found!")
    else:
        print(f"Loaded {len(existing_df)} existing feature vectors")
        
        # Determine which verdicts to re-extract
        if FIX_MODE == "empty_only":
            # Find verdicts with empty feature vectors
            mask = existing_df['feature_vector_json'] == '{}'
            verdicts_to_fix = existing_df[mask]
            print(f"\nFound {len(verdicts_to_fix)} verdicts with empty feature vectors")
            
        elif FIX_MODE == "error_only":
            # Find verdicts with potential parsing errors (less than 3 features)
            def count_features(json_str):
                try:
                    return len(json.loads(json_str))
                except:
                    return 0
            
            existing_df['feature_count'] = existing_df['feature_vector_json'].apply(count_features)
            mask = (existing_df['feature_count'] < 3) & (existing_df['indictment_facts'].notna())
            verdicts_to_fix = existing_df[mask]
            print(f"\nFound {len(verdicts_to_fix)} verdicts with errors (< 3 features)")
            
        elif FIX_MODE == "override_all":
            verdicts_to_fix = existing_df
            print(f"\n⚠️  WARNING: Will re-extract ALL {len(verdicts_to_fix)} verdicts!")
            response = input("Are you sure? Type 'yes' to continue: ")
            if response.lower() != 'yes':
                print("Cancelled.")
                verdicts_to_fix = existing_df[existing_df['verdict'] == '__NO_MATCH__']  # Empty
        
        if len(verdicts_to_fix) > 0:
            print("\nVerdicts to fix:")
            for idx, row in verdicts_to_fix.iterrows():
                verdict_id = row['verdict']
                facts_len = len(str(row['indictment_facts'])) if pd.notna(row['indictment_facts']) else 0
                current_features = row['feature_vector_json']
                print(f"  - {verdict_id}: facts={facts_len} chars, current={current_features[:50]}...")
            
            print(f"\n{'='*70}")
            print(f"Processing {len(verdicts_to_fix)} verdicts...")
            print(f"{'='*70}\n")
            
            # Process each verdict
            fixed_count = 0
            failed_count = 0
            
            for idx, row in verdicts_to_fix.iterrows():
                verdict_id = row['verdict']
                facts = row['indictment_facts']
                
                print(f"[{fixed_count + failed_count + 1}/{len(verdicts_to_fix)}] Processing {verdict_id}...")
                
                # Check if facts exist
                if pd.isna(facts) or str(facts).strip() == '':
                    print(f"  ⚠️  No indictment facts - skipping")
                    failed_count += 1
                    continue
                
                # Create VerdictFacts object
                verdict_facts = VerdictFacts(verdict=verdict_id, indictment_facts=facts)
                
                # Request features from GPT
                try:
                    feature_payload = request_features(
                        openai_client,
                        verdict_facts,
                        model=MODEL_NAME,
                        max_retries=MAX_RETRIES,
                        retry_delay=RETRY_DELAY_SECONDS,
                    )
                    
                    if feature_payload and len(feature_payload) > 0:
                        # Update in existing_df
                        new_json = json.dumps(feature_payload, ensure_ascii=False, sort_keys=True)
                        existing_df.loc[existing_df['verdict'] == verdict_id, 'feature_vector_json'] = new_json
                        print(f"  ✓ Extracted {len(feature_payload)} features")
                        fixed_count += 1
                    else:
                        print(f"  ⚠️  GPT returned empty features")
                        failed_count += 1
                        
                except Exception as e:
                    print(f"  ❌ Error: {e}")
                    failed_count += 1
                
                # Rate limiting
                if RETRY_DELAY_SECONDS > 0:
                    time.sleep(RETRY_DELAY_SECONDS)
            
            print(f"\n{'='*70}")
            print(f"Summary: Fixed={fixed_count}, Failed={failed_count}")
            print(f"{'='*70}\n")
            
            # Save updated feature vectors
            if fixed_count > 0:
                print("Saving updated feature vectors...")
                save_feature_vectors(
                    existing_df.to_dict(orient='records'),
                    FEATURE_VECTORS_PATH
                )
                print(f"✓ Saved to {FEATURE_VECTORS_PATH}")
                
                # Rebuild similarity database
                print("\nRebuilding similarity database...")
                similarity_base_df = prepare_similarity_frame(TARGET_PATH)
                gpt_similarity_df = build_similarity_dataset_with_gpt_features(
                    similarity_base_df,
                    existing_df,
                    FINAL_DATABASE_PATH,
                )
                print("✓ Done!")
        else:
            print("✓ No verdicts to fix!")
    
else:
    print(f"FIX_MODE is '{FIX_MODE}' - skipping. Change to 'empty_only', 'error_only', or 'override_all' to run.")


🔧 Running FIX MODE: empty_only
Loaded 68 existing feature vectors

Found 0 verdicts with empty feature vectors
✓ No verdicts to fix!


enrich manual features with gpt

In [14]:
import json
import pandas as pd
import time
from openai import OpenAI

# Reuse the client from your existing notebook if available, or initialize a new one
if 'openai_client' not in locals():
    openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ----------------------------------------------------------------------
# 1. System Prompt for Enrichment
# ----------------------------------------------------------------------
user_prompt = """
אתה עוזר משפטי מומחה במשפט הפלילי הישראלי.
המטרה שלך היא להעשיר וקטור מאפיינים קיים של תיק פלילי על בסיס עובדות כתב האישום.

יוצגו בפניך:
1. עובדות כתב האישום (Text).
2. רשימת מאפיינים שכבר חולצו באופן ידני (Existing Manual Features).

המשימה שלך:
הוצא פלט JSON בלבד המכיל מאפיינים **נוספים** או **משלימים** שלא נמצאים בפיצ'רים הקיימים.
אל תחזור על מידע שכבר קיים אלא אם כן יש לך תיקון מהותי.

חשוב מאוד:
במידה ויש פרטים בנוגע לעונש, להסדר טיעון או להכרעת הדין - אל תכלול אותם בפיצ'רים!
הפלט חייב להיות אובייקט JSON שטוח (Flat JSON) המכיל רק את התוספות.
"""

def enrich_single_case(facts: str, manual_features_str: str, model: str = "gpt-4-turbo") -> dict:
    """
    Sends facts + manual features to GPT and returns the merged dictionary (Manual + GPT).
    """
    # 1. Parse Manual Features using robust parsing
    try:
        if isinstance(manual_features_str, dict):
            manual_dict = manual_features_str
        elif isinstance(manual_features_str, str):
            # Try using the robust parse_feature_json function from the notebook
            manual_dict = parse_feature_json(manual_features_str)
            if not manual_dict:
                # If parse_feature_json returns empty, show the problematic string
                print(f"\n⚠️ Failed to parse manual features. Attempting basic fixes...")
                print(f"Problematic JSON (first 500 chars): {manual_features_str[:500]}")
                # Try basic cleanup
                cleaned = manual_features_str.replace("'", '"').replace("True", "true").replace("False", "false")
                manual_dict = json.loads(cleaned)
        else:
            manual_dict = {}
    except Exception as e:
        print(f"\n❌ Error parsing manual features: {e}")
        print(f"String that failed (first 500 chars): {manual_features_str[:500] if isinstance(manual_features_str, str) else manual_features_str}")
        if hasattr(e, 'pos') and isinstance(manual_features_str, str):
            print(f"Around error position {e.pos}:")
            print(f"...{manual_features_str[max(0, e.pos-50):min(len(manual_features_str), e.pos+100)]}...")
        # Don't continue with empty dict - raise the error so we can fix the data
        raise ValueError(f"Cannot parse manual features: {e}. Please fix the JSON in the source CSV.") from e

    # 2. Prepare Prompt
    manual_json_display = json.dumps(manual_dict, ensure_ascii=False, indent=2)
    
    user_content = f"""
    {user_prompt}
    
--- עובדות המקרה ---
{facts}

--- פיצ'רים ידניים קיימים ---
{manual_json_display}

"""

    # 3. Call GPT
    try:
        response = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0,
            max_tokens=2500,  # Increased to allow longer JSON responses
            response_format={"type": "json_object"}
        )
        content = response.choices[0].message.content
        gpt_additions = json.loads(content)
        
        # 4. Merge: Manual + GPT Additions
        # We start with manual and update with GPT (so GPT doesn't overwrite manual hard-facts unless intended, 
        # but usually we want to keep manual as truth for weights/amounts).
        # Actually, let's keep manual as the base truth.
        combined = manual_dict.copy()
        combined.update(gpt_additions)
        
        return combined

    except Exception as e:
        print(f"❌ Error in GPT enrichment: {e}")
        return manual_dict # Fallback to manual only


# ----------------------------------------------------------------------
# 2. Main Processing Function
# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# 2. Main Processing Function (Optimized for Unique Verdicts)
# ----------------------------------------------------------------------
def run_hybrid_experiment(manual_csv_path, facts_csv_path, output_path, skip_errors=False):
    """
    Args:
        manual_csv_path: Path to manual features CSV
        facts_csv_path: Path to facts CSV
        output_path: Path to save output
        skip_errors: If True, skip verdicts with parsing errors. If False, stop on first error.
    """
    print(f"🚀 Starting Hybrid Experiment...")
    print(f"   Reading Manual Features: {manual_csv_path}")
    print(f"   Reading Facts: {facts_csv_path}")
    print(f"   Skip errors mode: {skip_errors}")

    # Load DataFrames
    df_manual = pd.read_csv(manual_csv_path)
    df_facts = pd.read_csv(facts_csv_path)

    # Check column names for facts
    facts_col_1 = 'indicment_facts_1' if 'indicment_facts_1' in df_facts.columns else 'indictment_facts_1'
    facts_col_2 = 'indicment_facts_2' if 'indicment_facts_2' in df_facts.columns else 'indictment_facts_2'

    # Merge DataFrames to ensure alignment
    merged_df = pd.merge(
        df_manual, 
        df_facts[['verdict_1', 'verdict_2', facts_col_1, facts_col_2]], 
        on=['verdict_1', 'verdict_2'], 
        how='inner'
    )
    
    print(f"   Matched {len(merged_df)} pairs. Beginning processing...")

    # --- CACHE INITIALIZATION ---
    # Key: verdict_id, Value: enriched feature dictionary
    enrichment_cache = {} 
    failed_verdicts = []  # Track verdicts that failed to parse
    
    enriched_records = []
    
    for idx, row in merged_df.iterrows():
        if idx % 10 == 0:
            print(f"   Processing pair {idx}/{len(merged_df)} (Cache Size: {len(enrichment_cache)}, Failed: {len(failed_verdicts)})...")

        try:
            # --- Process Verdict 1 ---
            v1_id = row['verdict_1']
            
            # Check Cache for V1
            if v1_id in enrichment_cache:
                feat_vec_1 = enrichment_cache[v1_id]
            elif v1_id in failed_verdicts:
                if skip_errors:
                    print(f"   ⏭️ Skipping pair {idx} - Verdict 1 ({v1_id}) previously failed")
                    continue
                else:
                    raise ValueError(f"Verdict {v1_id} was marked as failed")
            else:
                # Not in cache -> Extract and Store
                try:
                    feat_vec_1 = enrich_single_case(
                        facts=row[facts_col_1],
                        manual_features_str=row['feature_vector_1']
                    )
                    enrichment_cache[v1_id] = feat_vec_1
                except Exception as e:
                    print(f"\n❌ Failed to process Verdict 1: {v1_id}")
                    failed_verdicts.append(v1_id)
                    if not skip_errors:
                        raise
                    print(f"   ⏭️ Skipping pair {idx} due to Verdict 1 error")
                    continue

            # --- Process Verdict 2 ---
            v2_id = row['verdict_2']
            
            # Check Cache for V2
            if v2_id in enrichment_cache:
                feat_vec_2 = enrichment_cache[v2_id]
            elif v2_id in failed_verdicts:
                if skip_errors:
                    print(f"   ⏭️ Skipping pair {idx} - Verdict 2 ({v2_id}) previously failed")
                    continue
                else:
                    raise ValueError(f"Verdict {v2_id} was marked as failed")
            else:
                # Not in cache -> Extract and Store
                try:
                    feat_vec_2 = enrich_single_case(
                        facts=row[facts_col_2],
                        manual_features_str=row['feature_vector_2']
                    )
                    enrichment_cache[v2_id] = feat_vec_2
                except Exception as e:
                    print(f"\n❌ Failed to process Verdict 2: {v2_id}")
                    failed_verdicts.append(v2_id)
                    if not skip_errors:
                        raise
                    print(f"   ⏭️ Skipping pair {idx} due to Verdict 2 error")
                    continue

            # --- Build Record ---
            enriched_records.append({
                'verdict_1': row['verdict_1'],
                'verdict_2': row['verdict_2'],
                'similarity_scale': row['similarity_scale'],
                'similarity_binary_0': row['similarity_binary_0'],
                'similarity_binary_1': row['similarity_binary_1'],
                'feature_vector_1': json.dumps(feat_vec_1, ensure_ascii=False),
                'feature_vector_2': json.dumps(feat_vec_2, ensure_ascii=False)
            })
            
        except Exception as e:
            print(f"\n❌ Error processing pair {idx}: {e}")
            if not skip_errors:
                raise

    # Save
    result_df = pd.DataFrame(enriched_records)
    result_df.to_csv(output_path, index=False)
    
    print(f"\n✅ Finished! Saved hybrid dataset to: {output_path}")
    print(f"   Successfully processed: {len(enriched_records)} pairs")
    print(f"   Failed verdicts: {len(failed_verdicts)}")
    if failed_verdicts:
        print(f"   Failed verdict IDs: {failed_verdicts}")
    
    return result_df

In [16]:
# --- RUN CONFIGURATION ---
# DOMAIN = "wep"  

# if DOMAIN == "drugs":
#     base_path="/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/"
# else:
#     base_path="/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon/"

# MANUAL_CSV = base_path + "similarity_database_fe.csv"  
# FACTS_CSV = base_path + "similarity_database_with_indicment_facts.csv"
# OUTPUT_CSV = base_path + "similarity_database_hybrid.csv"



# DOMAIN = "drugs"
# base_path="/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/"

# MANUAL_CSV = base_path + "similarity_database_fe_gpt_schema.csv"   # ← חדש!
# FACTS_CSV = base_path + "similarity_database_with_indicment_facts.csv"
# OUTPUT_CSV = base_path + "similarity_database_hybrid_full_gpt.csv"  # ← חדש!

DOMAIN = "wep"
base_path="/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon/"

MANUAL_CSV = base_path + "similarity_database_fe_gpt_schema.csv"   # ← חדש!
FACTS_CSV = base_path + "similarity_database_with_indicment_facts.csv"
OUTPUT_CSV = base_path + "similarity_database_hybrid_full_gpt.csv"  # ← חדש!


# Filter to only process changed verdicts
PROCESS_CHANGED_ONLY = False  # Set to False to process all verdicts

if PROCESS_CHANGED_ONLY:
    # Load comparison CSV to get changed verdict IDs
    comparison_csv_path = Path(base_path, "similarity_database_with_indicment_facts_comparison_dry_run.csv")
    
    if comparison_csv_path.exists():
        df_changes = pd.read_csv(comparison_csv_path)
        changed_verdicts = set(df_changes["verdict_id"].astype(str).str.strip())
        print(f"📂 Found {len(changed_verdicts)} changed verdicts in comparison CSV")
        
        # Filter manual CSV to only include pairs with changed verdicts
        df_manual_filtered = pd.read_csv(MANUAL_CSV)
        # Keep pairs where at least one verdict changed
        mask = (df_manual_filtered["verdict_1"].astype(str).str.strip().isin(changed_verdicts)) | \
              (df_manual_filtered["verdict_2"].astype(str).str.strip().isin(changed_verdicts))
        df_manual_filtered = df_manual_filtered[mask]
        
        # Save filtered manual CSV temporarily
        import tempfile
        temp_manual_csv = tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False)
        df_manual_filtered.to_csv(temp_manual_csv.name, index=False)
        MANUAL_CSV = temp_manual_csv.name
        print(f"   Filtered to {len(df_manual_filtered)} pairs with changed verdicts")
    else:
        print(f"⚠️  Comparison CSV not found at {comparison_csv_path}")
        print("   Processing all verdicts instead")
        PROCESS_CHANGED_ONLY = False

# Run
df_hybrid = run_hybrid_experiment(MANUAL_CSV, FACTS_CSV, OUTPUT_CSV)
print(df_hybrid.head())

🚀 Starting Hybrid Experiment...
   Reading Manual Features: /Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon/similarity_database_fe_gpt_schema.csv
   Reading Facts: /Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon/similarity_database_with_indicment_facts.csv
   Skip errors mode: False
   Matched 141 pairs. Beginning processing...
   Processing pair 0/141 (Cache Size: 0, Failed: 0)...
   Processing pair 10/141 (Cache Size: 18, Failed: 0)...
   Processing pair 20/141 (Cache Size: 32, Failed: 0)...
   Processing pair 30/141 (Cache Size: 45, Failed: 0)...
   Processing pair 40/141 (Cache Size: 54, Failed: 0)...
   Processing pair 50/141 (Cache Size: 59, Failed: 0)...
   Processing pair 60/141 (Cache Size: 66, Failed: 0)...
   Processing pair 70/141 (Cache Size: 74, Failed: 0)...
   Processing pair 80/141 (Cache Size: 78, Failed: 0)...
   Processing pair 90/141 (Cache Size: 82, Failed: 0)...
   Processing pair 100/141 (Cach